In [3]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
import os

csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]

print("CSV files found:")
for file in csv_files:
    print(file)

CSV files found:
employee_data.csv
faostat_india_crops_2015_2024.csv.csv
raw_data.csv


In [5]:
# Load the FAOSTAT dataset

file_path = "faostat_india_crops_2015_2024.csv.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 129
Columns: 15


In [6]:
# Basic dataset preview

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Dataset shape: (129, 15)

Column names:
['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']

First 5 records:


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,356,India,5510,Production,1921.02,"Cotton lint, ginned",2015,2015,t,5100850.0,A,Official value,NaN
1,QCL,Crops and livestock products,356,India,5510,Production,1921.02,"Cotton lint, ginned",2016,2016,t,5542000.0,A,Official value,NaN
2,QCL,Crops and livestock products,356,India,5510,Production,1921.02,"Cotton lint, ginned",2017,2017,t,5538090.0,A,Official value,NaN
3,QCL,Crops and livestock products,356,India,5510,Production,1921.02,"Cotton lint, ginned",2018,2018,t,5576850.0,A,Official value,NaN
4,QCL,Crops and livestock products,356,India,5510,Production,1921.02,"Cotton lint, ginned",2019,2019,t,4767140.0,A,Official value,NaN


In [7]:
# Data quality profiling

print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

print("\nNUMERIC SUMMARY")
print(df.describe())

DATA TYPES
Domain Code          object
Domain               object
Area Code (M49)       int64
Area                 object
Element Code          int64
Element              object
Item Code (CPC)     float64
Item                 object
Year Code             int64
Year                  int64
Unit                 object
Value               float64
Flag                 object
Flag Description     object
Note                float64
dtype: object

MISSING VALUES
Domain Code           0
Domain                0
Area Code (M49)       0
Area                  0
Element Code          0
Element               0
Item Code (CPC)       0
Item                  0
Year Code             0
Year                  0
Unit                  0
Value                 0
Flag                  0
Flag Description      0
Note                129
dtype: int64

DUPLICATE ROWS
0

NUMERIC SUMMARY
       Area Code (M49)  Element Code  Item Code (CPC)    Year Code  \
count            129.0    129.000000       129.000000   129.0

In [8]:
# Standardize column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

print(df.columns.tolist())

['domain_code', 'domain', 'area_code_m49', 'area', 'element_code', 'element', 'item_code_cpc', 'item', 'year_code', 'year', 'unit', 'value', 'flag', 'flag_description', 'note']


In [9]:
# Clean text fields

text_columns = [
    "domain",
    "area",
    "element",
    "item",
    "unit",
    "flag",
    "flag_description"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

print("Text fields cleaned.")

Text fields cleaned.


In [10]:
# Convert numeric columns safely

numeric_columns = [
    "area_code_m49",
    "element_code",
    "item_code_cpc",
    "year_code",
    "year",
    "value"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_columns].dtypes)

area_code_m49      int64
element_code       int64
item_code_cpc    float64
year_code          int64
year               int64
value            float64
dtype: object


In [11]:
# Check duplicates after cleaning

duplicate_count = df.duplicated().sum()

print("Duplicate rows after cleaning:", duplicate_count)

Duplicate rows after cleaning: 0


In [12]:
# Basic validity checks

print("Negative values:", (df["value"] < 0).sum())

print("Years outside selected range:",
      ((df["year"] < 2015) | (df["year"] > 2024)).sum())

print("Missing values in Value:", df["value"].isna().sum())

Negative values: 0
Years outside selected range: 0
Missing values in Value: 0


In [13]:
# Basic outlier investigation using IQR

Q1 = df["value"].quantile(0.25)
Q3 = df["value"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["value"] < lower_bound) |
    (df["value"] > upper_bound)
]

print("Potential outliers:", len(outliers))

display(
    outliers[
        ["item", "year", "element", "value", "unit"]
    ].head(20)
)

Potential outliers: 20


,item,year,element,value,unit
41,Rice,2015,Production,1.582550e+08,t
44,Rice,2016,Production,1.639474e+08,t
47,Rice,2017,Production,1.657962e+08,t
50,Rice,2018,Production,1.764937e+08,t
53,Rice,2019,Production,1.750715e+08,t
56,Rice,2020,Production,1.827029e+08,t
59,Rice,2021,Production,1.952414e+08,t
62,Rice,2022,Production,1.934733e+08,t
65,Rice,2023,Production,2.077536e+08,t
68,Rice,2024,Production,2.178679e+08,t


In [14]:
# Save cleaned dataset

cleaned_file = "faostat_india_crops_2015_2024_cleaned.csv"

df.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved as:", cleaned_file)
print("Final shape:", df.shape)

Cleaned dataset saved as: faostat_india_crops_2015_2024_cleaned.csv
Final shape: (129, 15)
